# Análise dos resultados do Citizen Assist AI

Este notebook lê os resultados gerados por `python src/avaliacao.py` (pasta `docs/resultados/`)
e calcula os indicadores de acerto e de tempo de consulta. Rode a avaliação antes de executar as células.

In [1]:
import pandas as pd
from pathlib import Path

PASTA = Path("..") / "docs" / "resultados"
testes = pd.read_csv(PASTA / "resultados_testes.csv", sep=";")
testes.head()

,conjunto,n,tipo,pergunta,esperado,id_esperado,obtido,id_obtido,pontuacao,acerto,tempo_ms
0,teste,1,direta,Como faço a nova carteira de identidade?,respondida,DOC01,respondida,DOC01,0.9551,acerto,0.1467
1,teste,2,direta,Como consultar a situação do meu CPF?,respondida,DOC02,respondida,DOC02,0.6806,acerto,0.1317
2,teste,3,direta,Onde baixo o título de eleitor digital?,respondida,DOC03,respondida,DOC03,1.0000,acerto,0.1573
3,teste,4,direta,Como pedir seguro-desemprego?,respondida,SER02,respondida,SER02,0.8753,acerto,0.1597
4,teste,5,direta,Qual o telefone do INSS?,respondida,SER04,respondida,SER04,0.8121,acerto,0.1146


## Taxa de acerto e tempo de consulta por conjunto

In [2]:
resumo = testes.groupby("conjunto").agg(
    perguntas=("n", "count"),
    acertos=("acerto", lambda s: (s == "acerto").sum()),
    tempo_medio_ms=("tempo_ms", "mean"),
    tempo_max_ms=("tempo_ms", "max"),
)
resumo["taxa_acerto_%"] = (100 * resumo["acertos"] / resumo["perguntas"]).round(1)
resumo.round(3)

,perguntas,acertos,tempo_medio_ms,tempo_max_ms,taxa_acerto_%
conjunto,,,,,
teste,24,24,0.280,0.792,100.0
validacao,12,9,0.224,0.454,75.0


## Acertos por tipo de pergunta

In [3]:
por_tipo = (testes.assign(ok=testes["acerto"] == "acerto")
            .pivot_table(index="tipo", columns="conjunto", values="ok", aggfunc="mean") * 100)
por_tipo.round(1)

conjunto,teste,validacao
tipo,,
direta,100.0,NaN
erro_digitacao,100.0,NaN
fora_do_tema,100.0,100.0
limitacao,100.0,0.0
privacidade,100.0,NaN
reformulada,100.0,77.8


## Erros: onde o assistente falhou

In [4]:
testes.loc[testes["acerto"] == "erro", ["conjunto", "pergunta", "esperado", "id_esperado", "obtido", "id_obtido", "pontuacao"]]

,conjunto,pergunta,esperado,id_esperado,obtido,id_obtido,pontuacao
28,validacao,"Meu pai quer dar entrada na aposentadoria, por...",respondida,SER04,sem_resposta,NaN,0.1439
29,validacao,Pedi um produto pelo telefone e quero cancelar,respondida,DIR01,sem_resposta,NaN,0.1647
35,validacao,documento,ambigua,NaN,respondida,DOC01,0.6667


## Comparação dos modelos de resposta

In [5]:
modelos = pd.read_csv(PASTA / "comparacao_modelos.csv", sep=";")
criterios = ["cita_link_da_fonte", "cita_orgao_da_fonte", "mostra_data_de_verificacao",
             "aviso_quando_necessario", "organizado_em_topicos"]
tabela = modelos.groupby("modelo")[criterios].mean().T * 100
tabela.loc["palavras_por_frase"] = modelos.groupby("modelo")["palavras_por_frase"].mean()
tabela.loc["indice_flesch"] = modelos.groupby("modelo")["indice_flesch"].mean()
tabela.round(1)

modelo,A,B
cita_link_da_fonte,100.0,100.0
cita_orgao_da_fonte,0.0,100.0
mostra_data_de_verificacao,0.0,100.0
aviso_quando_necessario,47.1,100.0
organizado_em_topicos,0.0,100.0
palavras_por_frase,12.4,10.1
indice_flesch,60.4,59.5


## Interpretação

- No conjunto de **teste**, usado para calibrar o limiar e os pesos da busca, o acerto foi de 100%.
  Esse número é otimista, porque o sistema foi ajustado olhando para essas perguntas.
- No conjunto de **validação**, escrito depois do ajuste, o acerto caiu para 75%. Esse é o número
  mais honesto. Os erros vieram de palavras do dia a dia que não estão na base ("cancelar",
  "dar entrada") e de uma palavra genérica ("documento").
- O tempo médio de consulta ficou abaixo de 1 ms, porque a busca usa um índice invertido (dicionário)
  e seleciona as melhores respostas com um heap.
- O Modelo B cumpre 100% do checklist de transparência (fonte, órgão, data e aviso), contra 29,4%
  do Modelo A, com legibilidade semelhante.